# Ingest and Vectorize Zoning Laws

### Tech:

To extract raw text from pdfs use PyPDF2 or PyMuPDF/ for docs: python-docx

Langchain to handle splitting and chunking logic

ChromaDB local vector store in the directory

Embeddings from OpenAI or another service



### Notes:

Semantic Chunking is preferred over Recursive Character splitter (standard) with overlap of some tokens based on semantics. To do that, you need embedding API and then do the semantic chunking

### Read the orlando zoning files

In [12]:
import os
from docx import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

# Set environment variable to fix ChromaDB config issue
os.environ["CHROMA_SERVER_NOFILE"] = "1024"

from langchain_chroma import Chroma

from dotenv import load_dotenv
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

# Verify API key is loaded
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please check your .env file.")
print("✓ API key loaded successfully")

ConfigError: unable to infer type for attribute "chroma_server_nofile"

In [5]:
orlando_zoning_laws_path = "data/orlando-zoning-laws"
files = os.listdir(orlando_zoning_laws_path)
for f in files:
    print(f)

Chapter_14___PROPERTY_MAINTENANCE_CODE.docx
Chapter_58___ZONING_DISTRICTS_AND_USES.docx
Chapter_60___SUBDIVISION_AND_LANDSCAPING.docx
Chapter_61___ROADWAY_DESIGN_AND_ACCESS_MANAGEMENT.docx
Chapter_63___ENVIRONMENTAL_PROTECTION.docx
Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx
Chapter_66___DEFINITIONS.docx
Chapter_67___AFFORDABLE_HOUSING.docx
Orlando-Zoning-Chapter-Descriptions.xlsx
OrlandoFLCodeofOrdinancesEXPORT20251209.xlsx


In [6]:
# Get only .docx files
docx_files = [f for f in os.listdir(orlando_zoning_laws_path) if f.endswith('.docx')]

# Dictionary to store text from each document
documents = {}

for filename in docx_files:
    filepath = os.path.join(orlando_zoning_laws_path, filename)
    doc = Document(filepath)
    
    # Extract all paragraphs
    text = "\n".join([para.text for para in doc.paragraphs])
    documents[filename] = text
    
    print(f"✓ Loaded: {filename} ({len(text)} characters)")

print(f"\nTotal documents loaded: {len(documents)}")

✓ Loaded: Chapter_14___PROPERTY_MAINTENANCE_CODE.docx (66196 characters)
✓ Loaded: Chapter_58___ZONING_DISTRICTS_AND_USES.docx (595020 characters)
✓ Loaded: Chapter_60___SUBDIVISION_AND_LANDSCAPING.docx (110424 characters)
✓ Loaded: Chapter_61___ROADWAY_DESIGN_AND_ACCESS_MANAGEMENT.docx (171587 characters)
✓ Loaded: Chapter_63___ENVIRONMENTAL_PROTECTION.docx (98777 characters)
✓ Loaded: Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx (445724 characters)
✓ Loaded: Chapter_66___DEFINITIONS.docx (175223 characters)
✓ Loaded: Chapter_67___AFFORDABLE_HOUSING.docx (58442 characters)

Total documents loaded: 8


### Split and Chunk the orlando zoning laws

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Chunk all documents
all_chunks = []

for filename, text in documents.items():
    chunks = splitter.create_documents([text])
    for i, chunk in enumerate(chunks):
        chunk.metadata = {
            "source": filename,
            "chunk_id": i,
            "total_chunks": len(chunks),
            "char_count": len(chunk.page_content),
            "chapter": filename.split("___")[0].replace("_", " "),
        }
    all_chunks.extend(chunks)
    print(f"✓ {filename}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

✓ Chapter_14___PROPERTY_MAINTENANCE_CODE.docx: 107 chunks
✓ Chapter_58___ZONING_DISTRICTS_AND_USES.docx: 907 chunks
✓ Chapter_60___SUBDIVISION_AND_LANDSCAPING.docx: 171 chunks
✓ Chapter_61___ROADWAY_DESIGN_AND_ACCESS_MANAGEMENT.docx: 257 chunks
✓ Chapter_63___ENVIRONMENTAL_PROTECTION.docx: 138 chunks
✓ Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx: 673 chunks
✓ Chapter_66___DEFINITIONS.docx: 232 chunks
✓ Chapter_67___AFFORDABLE_HOUSING.docx: 89 chunks

Total chunks: 2574


In [8]:
# Check a sample chunk
print(all_chunks[0].page_content[:500])
print("\n--- Metadata ---")
print(all_chunks[0].metadata)

Chapter 14 PROPERTY MAINTENANCE CODE

ARTICLE I. PURPOSE, ADMINISTRATION, APPLICATION AND ENFORCEMENT

--- Metadata ---
{'source': 'Chapter_14___PROPERTY_MAINTENANCE_CODE.docx', 'chunk_id': 0, 'total_chunks': 107, 'char_count': 101, 'chapter': 'Chapter 14'}


### Generate Embeddings

In [9]:
# Recommended - explicitly pass API key
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=openai_api_key
)

# Test it
# test_vector = embeddings.embed_query("zoning regulations in Orlando")
# print(f"Vector dimensions: {len(test_vector)}")

In [10]:
# Create vector store from your chunks
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"  # saves locally
)

print(f"✓ Vector store created with {vectorstore._collection.count()} documents")

NameError: name 'Chroma' is not defined